# Head 1: Condition Classifier Training

Training a 5-class mental health condition classifier (Normal, Depression, Anxiety, Stress, Suicidal) on a shared MentalBERT backbone using combined Mukherjee and Dreaddit data.

This notebook is one component of a two-head classification pipeline. Head 2 (Cause classifier on CAMS) is trained separately.

**Input data**: Pre-split CSVs from the data preparation step. This notebook does not re-clean or re-split anything.

**Output**: A trained checkpoint, label mapping, test evaluation results, and a confound check.

## Section 1: Environment Check

In [1]:
import os
import sys
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("SECTION 1: ENVIRONMENT CHECK")

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU detected: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected.")
    print("Enable GPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Load HF token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        print(f"WARNING: Could not load HF_TOKEN: {e}")
        print("MentalBERT requires authentication. Add HF_TOKEN as a Kaggle Secret.")

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

SECTION 1: ENVIRONMENT CHECK
GPU detected: Tesla T4
VRAM: 14.6 GB
PyTorch version: 2.10.0+cu128
Device: cuda
HF_TOKEN loaded from Kaggle Secrets.
Checkpoint directory: /kaggle/working/checkpoints


## Section 2: Load and Inspect Data

Load the pre-split training, validation, and test CSVs. Print row counts, class distributions, and source dataset breakdowns for each file to confirm the data arrived correctly before any training begins.

In [2]:
print("SECTION 2: LOAD AND INSPECT DATA")

DATA_DIR = "/kaggle/input/datasets/daltonkhatri/head1-traning"

train_df = pd.read_csv(os.path.join(DATA_DIR, "head1_train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "head1_val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "head1_test.csv"))

for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}: {len(df):,d} rows")
    print(f"  Columns: {list(df.columns)}")
    print(f"\n  condition_label distribution:")
    counts = df["condition_label"].value_counts()
    for label, count in counts.items():
        pct = 100 * count / len(df)
        print(f"    {label:20s}  {count:>7,d}  ({pct:5.1f}%)")
    print(f"\n  source_dataset breakdown:")
    src_counts = df["source_dataset"].value_counts()
    for src, count in src_counts.items():
        pct = 100 * count / len(df)
        print(f"    {src:20s}  {count:>7,d}  ({pct:5.1f}%)")

SECTION 2: LOAD AND INSPECT DATA

Train: 41,582 rows
  Columns: ['cleaned_text', 'raw_text', 'condition_label', 'source_dataset', 'subreddit']

  condition_label distribution:
    Normal                 15,504  ( 37.3%)
    Depression             11,394  ( 27.4%)
    Suicidal                8,951  ( 21.5%)
    Anxiety                 4,254  ( 10.2%)
    Stress                  1,479  (  3.6%)

  source_dataset breakdown:
    mukherjee              38,759  ( 93.2%)
    dreaddit                2,823  (  6.8%)

Validation: 5,198 rows
  Columns: ['cleaned_text', 'raw_text', 'condition_label', 'source_dataset', 'subreddit']

  condition_label distribution:
    Normal                  1,938  ( 37.3%)
    Depression              1,425  ( 27.4%)
    Suicidal                1,119  ( 21.5%)
    Anxiety                   531  ( 10.2%)
    Stress                    185  (  3.6%)

  source_dataset breakdown:
    mukherjee               4,845  ( 93.2%)
    dreaddit                  353  (  6.8%)

Te

## Section 3: Label Encoding

Map the five condition class names to integer indices. This mapping is stored explicitly and saved to disk so it can be reused later without guessing the order.

In [4]:
print("SECTION 3: LABEL ENCODING")

LABEL_TO_ID = {
    "Normal": 0,
    "Depression": 1,
    "Anxiety": 2,
    "Stress": 3,
    "Suicidal": 4,
}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

print("\nLabel mapping:")
for label, idx in LABEL_TO_ID.items():
    print(f"  {label} -> {idx}")

# Apply encoding to all splits
for df in [train_df, val_df, test_df]:
    df["label"] = df["condition_label"].map(LABEL_TO_ID)

# Verify no nulls from unmapped labels
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    n_null = df["label"].isna().sum()
    if n_null > 0:
        print(f"  WARNING: {name} has {n_null} unmapped labels")
    else:
        print(f"  {name}: all labels mapped successfully")

# Save the mapping
mapping_path = os.path.join(CHECKPOINT_DIR, "label_mapping.json")
with open(mapping_path, "w") as f:
    json.dump({str(k): v for k, v in ID_TO_LABEL.items()}, f, indent=2)
print(f"\nLabel mapping saved to {mapping_path}")

SECTION 3: LABEL ENCODING

Label mapping:
  Normal -> 0
  Depression -> 1
  Anxiety -> 2
  Stress -> 3
  Suicidal -> 4
  Train: all labels mapped successfully
  Val: all labels mapped successfully
  Test: all labels mapped successfully

Label mapping saved to /kaggle/working/checkpoints/label_mapping.json


## Section 4: Domain-Balanced Batch Sampler

Mukherjee has ~93% of training rows and Dreaddit has ~7%. If batches are sampled randomly, most batches will be almost entirely Mukherjee, and the model might quietly learn to recognize which dataset a post came from instead of learning the actual mental health condition.

This custom sampler ensures every training batch contains roughly equal numbers of rows from each source. The larger source (Mukherjee) determines how many batches make up one epoch. The smaller source (Dreaddit) is repeated as many times as needed to fill its half of every batch.

In [5]:
print("SECTION 4: DOMAIN-BALANCED BATCH SAMPLER")


class DomainBalancedSampler(Sampler):

    def __init__(self, source_labels, batch_size):
        super().__init__()
        self.batch_size = batch_size
        self.half_batch = batch_size // 2

        # Separate row indices by source dataset
        self.muk_indices = [
            i for i, s in enumerate(source_labels) if s == "mukherjee"
        ]
        self.dread_indices = [
            i for i, s in enumerate(source_labels) if s == "dreaddit"
        ]

        # Number of batches set by the larger source so all its data is seen
        n_muk_batches = len(self.muk_indices) // self.half_batch
        n_dread_batches = len(self.dread_indices) // self.half_batch
        self.n_batches = max(n_muk_batches, n_dread_batches)

        # Report sampling rates so they are visible, not hidden
        muk_samples = self.n_batches * self.half_batch
        dread_samples = self.n_batches * self.half_batch
        muk_repeats = muk_samples / max(len(self.muk_indices), 1)
        dread_repeats = dread_samples / max(len(self.dread_indices), 1)

        print(f"\n  DomainBalancedSampler initialized:")
        print(f"    Mukherjee:  {len(self.muk_indices):,d} unique indices, "
              f"sampled {muk_samples:,d} times per epoch ({muk_repeats:.1f}x)")
        print(f"    Dreaddit:   {len(self.dread_indices):,d} unique indices, "
              f"sampled {dread_samples:,d} times per epoch ({dread_repeats:.1f}x)")
        print(f"    Batches per epoch: {self.n_batches:,d}")
        print(f"    Effective samples per epoch: {self.n_batches * batch_size:,d}")

    def _extend_and_shuffle(self, indices, target_length):
        """Shuffle the index list and repeat it to reach the target length."""
        pool = indices.copy()
        random.shuffle(pool)
        repeats_needed = (target_length // len(pool)) + 1
        extended = (pool * repeats_needed)[:target_length]
        return extended

    def __iter__(self):
        target = self.n_batches * self.half_batch
        muk_pool = self._extend_and_shuffle(self.muk_indices, target)
        dread_pool = self._extend_and_shuffle(self.dread_indices, target)

        # Build batches: half from each source, then shuffle within the batch
        all_indices = []
        for i in range(self.n_batches):
            start = i * self.half_batch
            end = start + self.half_batch
            batch = muk_pool[start:end] + dread_pool[start:end]
            random.shuffle(batch)
            all_indices.extend(batch)

        return iter(all_indices)

    def __len__(self):
        return self.n_batches * self.batch_size


# Instantiate to confirm it works and print sampling rates
source_labels_train = train_df["source_dataset"].tolist()
_ = DomainBalancedSampler(source_labels_train, batch_size=16)

SECTION 4: DOMAIN-BALANCED BATCH SAMPLER

  DomainBalancedSampler initialized:
    Mukherjee:  38,759 unique indices, sampled 38,752 times per epoch (1.0x)
    Dreaddit:   2,823 unique indices, sampled 38,752 times per epoch (13.7x)
    Batches per epoch: 4,844
    Effective samples per epoch: 77,504


## Section 5: Dataset and DataLoader

A PyTorch Dataset that tokenizes text on the fly using the MentalBERT tokenizer. Max token length is 256 rather than 512 because most Reddit posts are well under 256 tokens and doubling to 512 doubles memory usage without meaningful gains.

The training DataLoader uses the domain-balanced sampler from Section 4. Validation and test DataLoaders use standard sequential order since balance is not needed during evaluation.

In [6]:
print("SECTION 5: DATASET AND DATALOADER")

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32


class ConditionDataset(Dataset):
    """ PyTorch Dataset that tokenizes cleaned text and returns model inputs."""

    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts = dataframe["cleaned_text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]) if self.texts[idx] is not None else ""
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


# Load tokenizer
print("Loading MentalBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "mental/mental-bert-base-uncased", token=HF_TOKEN
)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

# Create datasets
train_dataset = ConditionDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = ConditionDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = ConditionDataset(test_df, tokenizer, MAX_LENGTH)

print(f"Train dataset: {len(train_dataset):,d} samples")
print(f"Val dataset:   {len(val_dataset):,d} samples")
print(f"Test dataset:  {len(test_dataset):,d} samples")

# Create data loaders
train_sampler = DomainBalancedSampler(source_labels_train, TRAIN_BATCH_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"\nTrain loader: {len(train_loader):,d} batches (batch_size={TRAIN_BATCH_SIZE})")
print(f"Val loader:   {len(val_loader):,d} batches (batch_size={EVAL_BATCH_SIZE})")
print(f"Test loader:  {len(test_loader):,d} batches (batch_size={EVAL_BATCH_SIZE})")

# Quick sanity check: pull one batch and verify shapes
sample_batch = next(iter(val_loader))
print(f"\nSample batch shapes:")
print(f"  input_ids:      {list(sample_batch['input_ids'].shape)}")
print(f"  attention_mask:  {list(sample_batch['attention_mask'].shape)}")
print(f"  label:           {list(sample_batch['label'].shape)}")

SECTION 5: DATASET AND DATALOADER
Loading MentalBERT tokenizer...


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded. Vocab size: 30522
Train dataset: 41,582 samples
Val dataset:   5,198 samples
Test dataset:  5,198 samples

  DomainBalancedSampler initialized:
    Mukherjee:  38,759 unique indices, sampled 38,752 times per epoch (1.0x)
    Dreaddit:   2,823 unique indices, sampled 38,752 times per epoch (13.7x)
    Batches per epoch: 4,844
    Effective samples per epoch: 77,504

Train loader: 4,844 batches (batch_size=16)
Val loader:   163 batches (batch_size=32)
Test loader:  163 batches (batch_size=32)

Sample batch shapes:
  input_ids:      [32, 256]
  attention_mask:  [32, 256]
  label:           [32]


## Section 6: Model

Load MentalBERT as the backbone and add a classification head on top. The head is a dropout layer followed by a linear layer mapping from MentalBERT's 768-dimensional hidden state to 5 condition classes. The CLS token's pooled representation is used as the input to the classification head.

In [7]:
print("SECTION 6: MODEL")

class ConditionClassifier(nn.Module):
    """ MentalBERT backbone with a dropout + linear classification head."""

    def __init__(self, backbone, num_classes=5, dropout_rate=0.1):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask
        )
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits


# Load MentalBERT backbone
print("Loading MentalBERT backbone...")
backbone = AutoModel.from_pretrained(
    "mental/mental-bert-base-uncased", token=HF_TOKEN
)
print("MentalBERT loaded successfully.")

# Build the classifier
model = ConditionClassifier(backbone, num_classes=5, dropout_rate=0.1)
model = model.to(device)

# Print parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,d}")
print(f"Trainable parameters: {trainable_params:,d}")

SECTION 6: MODEL
Loading MentalBERT backbone...


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your d

MentalBERT loaded successfully.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Total parameters:     109,486,085
Trainable parameters: 109,486,085


## Section 7: Loss Function

Use weighted cross-entropy to handle class imbalance. The weights are computed using sklearn's `compute_class_weight` with strategy "balanced", which assigns higher weight to underrepresented classes (especially Stress at 3.6% of training data). This works alongside the domain-balanced sampler: the sampler handles source imbalance at the batch level, while the loss weights handle class imbalance at the gradient level.

In [8]:
print("SECTION 7: LOSS FUNCTION")

# Compute class weights from training label distribution
train_labels_np = train_df["label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(LABEL_TO_ID.values())),
    y=train_labels_np,
)

print("\nComputed class weights:")
for idx, weight in enumerate(class_weights):
    label_name = ID_TO_LABEL[idx]
    count = (train_labels_np == idx).sum()
    print(f"  {label_name:15s} (n={count:>6,d}): weight = {weight:.4f}")

weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
print(f"\nWeighted CrossEntropyLoss created on {device}")

SECTION 7: LOSS FUNCTION

Computed class weights:
  Normal          (n=15,504): weight = 0.5364
  Depression      (n=11,394): weight = 0.7299
  Anxiety         (n= 4,254): weight = 1.9550
  Stress          (n= 1,479): weight = 5.6230
  Suicidal        (n= 8,951): weight = 0.9291

Weighted CrossEntropyLoss created on cuda


# Section 7.5: Training Tracker Initialization

In [9]:
history = {
    "epoch": [],
    "train_loss": [],
    "val_macro_f1": [],
    "val_accuracy": [],
    "val_stress_f1": [],
    "val_normal_f1": [],
    "val_depression_f1": [],
    "val_anxiety_f1": [],
    "val_suicidal_f1": [],
}
print("Training history tracker initialized.")

Training history tracker initialized.


## Section 8: Training Loop

Train for up to 10 epochs with AdamW optimizer and linear warmup over the first 10% of total training steps. The best model checkpoint (by validation macro F1) is saved after each improving epoch.

Training stops early if validation macro F1 does not improve for 3 consecutive epochs to avoid wasting compute on overfitting.

Stress F1 is monitored separately every epoch because Stress is the most underrepresented class (3.6% of training data). A warning is printed if it drops below 0.40 at any point.

In [11]:
# Label names in index order, used throughout evaluation
LABEL_NAMES = [ID_TO_LABEL[i] for i in range(len(ID_TO_LABEL))]


def evaluate_model(model, dataloader, device):
    """Run inference on a dataloader and return predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)


def compute_metrics(preds, labels):
    """Compute macro F1, per-class metrics, confusion matrix, and accuracy."""
    macro_f1 = f1_score(labels, preds, average="macro")
    accuracy = accuracy_score(labels, preds)
    report_dict = classification_report(
        labels, preds, target_names=LABEL_NAMES,
        output_dict=True, zero_division=0
    )
    report_str = classification_report(
        labels, preds, target_names=LABEL_NAMES, zero_division=0
    )
    cm = confusion_matrix(labels, preds)

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "report_dict": report_dict,
        "report_str": report_str,
        "confusion_matrix": cm,
    }


def print_epoch_metrics(epoch, train_loss, metrics):
    """Print a formatted summary of one epoch's results."""
    print(f"\n  Epoch {epoch} Summary:")
    print(f"    Train Loss:    {train_loss:.4f}")
    print(f"    Val Macro F1:  {metrics['macro_f1']:.4f}")
    print(f"    Val Accuracy:  {metrics['accuracy']:.4f}")
    print(f"\n    Per-class results:")
    print(f"    {'Class':15s} {'F1':>8s} {'Precision':>10s} {'Recall':>8s}")
    print(f"    {'-' * 43}")
    for label_name in LABEL_NAMES:
        cls = metrics["report_dict"][label_name]
        print(
            f"    {label_name:15s} {cls['f1-score']:8.4f} "
            f"{cls['precision']:10.4f} {cls['recall']:8.4f}"
        )
# record history
    history["epoch"].append(epoch)
    history["train_loss"].append(avg_train_loss)
    history["val_macro_f1"].append(val_metrics["macro_f1"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    for cls in ["Stress", "Normal", "Depression", "Anxiety", "Suicidal"]:
        history[f"val_{cls.lower()}_f1"].append(
            val_metrics["report_dict"][cls]["f1-score"]
        )
        
    # Stress specific warning
    stress_f1 = metrics["report_dict"]["Stress"]["f1-score"]
    if stress_f1 < 0.40:
        print(
            f"\n    WARNING: Stress F1 ({stress_f1:.4f}) is below 0.40. "
            f"The model is struggling with the most underrepresented class."
        )

In [ ]:
print("SECTION 8: TRAINING LOOP")

# Hyperparameters
LEARNING_RATE = 2e-5
MAX_EPOCHS = 10
WARMUP_PROPORTION = 0.1
PATIENCE = 3

# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Scheduler with linear warmup
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_PROPORTION)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay=0.01)")
print(f"Total training steps: {total_steps:,d}")
print(f"Warmup steps: {warmup_steps:,d} ({WARMUP_PROPORTION * 100:.0f}%)")
print(f"Max epochs: {MAX_EPOCHS}")
print(f"Early stopping patience: {PATIENCE}")
print(f"Gradient clipping: max_norm=1.0")

best_macro_f1 = 0.0
patience_counter = 0
best_epoch = 0
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_head1_checkpoint.pt")

for epoch in range(1, MAX_EPOCHS + 1):
    print(f"\n{'=' * 60}")
    print(f"EPOCH {epoch}/{MAX_EPOCHS}")
    print(f"{'=' * 60}")

    # Training phase
    model.train()
    total_loss = 0.0
    n_batches = 0

    progress = tqdm(train_loader, desc=f"Training epoch {epoch}", leave=True)
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches += 1
        progress.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_loss / n_batches

    # Validation phase
    val_preds, val_labels_arr = evaluate_model(model, val_loader, device)
    val_metrics = compute_metrics(val_preds, val_labels_arr)

    # Print epoch results
    print_epoch_metrics(epoch, avg_train_loss, val_metrics)

    # Check for new best
    current_f1 = val_metrics["macro_f1"]
    if current_f1 > best_macro_f1:
        best_macro_f1 = current_f1
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_f1": best_macro_f1,
                "label_mapping": ID_TO_LABEL,
            },
            checkpoint_path,
        )
        print(f"\n    NEW BEST checkpoint saved (macro F1: {best_macro_f1:.4f})")
    else:
        patience_counter += 1
        print(
            f"\n    No improvement. Best: {best_macro_f1:.4f} at epoch {best_epoch}. "
            f"Patience: {patience_counter}/{PATIENCE}"
        )

    if patience_counter >= PATIENCE:
        print(
            f"\n  EARLY STOPPING triggered after {PATIENCE} epochs "
            f"without improvement."
        )
        print(f"  Best validation macro F1: {best_macro_f1:.4f} at epoch {best_epoch}")
        break

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE")
print(f"Best epoch: {best_epoch}, Best val macro F1: {best_macro_f1:.4f}")
print(f"Checkpoint saved to: {checkpoint_path}")
print(f"{'=' * 60}")

SECTION 8: TRAINING LOOP
Optimizer: AdamW (lr=2e-05, weight_decay=0.01)
Total training steps: 48,440
Warmup steps: 4,844 (10%)
Max epochs: 10
Early stopping patience: 3
Gradient clipping: max_norm=1.0

EPOCH 1/10


Training epoch 1:   0%|          | 0/4844 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/163 [00:00<?, ?it/s]


  Epoch 1 Summary:
    Train Loss:    0.3778
    Val Macro F1:  0.7603
    Val Accuracy:  0.7978

    Per-class results:
    Class                 F1  Precision   Recall
    -------------------------------------------
    Normal            0.9236     0.9440   0.9040
    Depression        0.7032     0.8177   0.6168
    Anxiety           0.7769     0.6922   0.8851
    Stress            0.6625     0.5390   0.8595
    Suicidal          0.7355     0.6860   0.7927

    NEW BEST checkpoint saved (macro F1: 0.7603)

EPOCH 2/10


Training epoch 2:   0%|          | 0/4844 [00:00<?, ?it/s]

...


  Epoch 4 Summary:
    Train Loss:    0.0421
    Val Macro F1:  0.8100
    Val Accuracy:  0.8419

    Per-class results:
    Class                 F1  Precision   Recall
    -------------------------------------------
    Normal            0.9484     0.9489   0.9479
    Depression        0.7771     0.7966   0.7586
    Anxiety           0.8658     0.8691   0.8625
    Stress            0.7062     0.6749   0.7405
    Suicidal          0.7524     0.7345   0.7712

    No improvement. Best: 0.8125 at epoch 3. Patience: 1/3

EPOCH 5/10


Training epoch 5:   0%|          | 0/4844 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/163 [00:00<?, ?it/s]


  Epoch 5 Summary:
    Train Loss:    0.0268
    Val Macro F1:  0.8126
    Val Accuracy:  0.8382

    Per-class results:
    Class                 F1  Precision   Recall
    -------------------------------------------
    Normal            0.9508     0.9644   0.9376
    Depression        0.7692     0.7598   0.7789
    Anxiety           0.8598     0.8427   0.8776
    Stress            0.7435     0.7208   0.7676
    Suicidal          0.7399     0.7452   0.7346

    NEW BEST checkpoint saved (macro F1: 0.8126)

EPOCH 6/10


Training epoch 6:   0%|          | 0/4844 [00:00<?, ?it/s]

## Section 9: Test Set Evaluation

Load the best saved checkpoint and evaluate on the held-out test set, which was not seen during training or validation. This gives an unbiased estimate of model performance.

Results are compared against reference baselines: random guessing (~20% macro F1 for 5 classes) and typical fine-tuned BERT performance on similar tasks (65 to 80% macro F1).

In [ ]:
print("SECTION 9: TEST SET EVALUATION")


# Load best checkpoint
print(f"Loading best checkpoint from epoch {best_epoch}...")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Checkpoint loaded. Val macro F1 was: {checkpoint['val_macro_f1']:.4f}")

# Run on test set
test_preds, test_true = evaluate_model(model, test_loader, device)
test_metrics = compute_metrics(test_preds, test_true)

# Print results
print(f"\nTest Set Results:")
print(f"  Macro F1:  {test_metrics['macro_f1']:.4f}")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"\nClassification Report:")
print(test_metrics["report_str"])
print(f"Confusion Matrix:")
print(f"  Rows = true labels, Columns = predicted labels")
print(f"  Label order: {LABEL_NAMES}")
print(test_metrics["confusion_matrix"])

# Baseline comparison
print(f"\nBaseline Comparison:")
print(f"  Random baseline (5 classes):   ~0.2000 macro F1")
print(f"  Strong fine-tuned BERT:        ~0.65 to 0.80 macro F1")
print(f"  This model:                     {test_metrics['macro_f1']:.4f} macro F1")

if test_metrics["macro_f1"] < 0.50:
    print(f"\n  NOTE: Test macro F1 ({test_metrics['macro_f1']:.4f}) is below 0.50.")
    print(f"  This is below the expected range for a fine-tuned model.")
    print(f"  Possible causes to investigate:")
    print(f"    1. Label encoding mismatch between training and evaluation")
    print(f"    2. Data leakage or contamination in the splits")
    print(f"    3. Domain balanced sampler creating too much repetition")
    print(f"    4. Learning rate too high or too low")
    print(f"    5. Not enough training epochs before early stopping")

# Save results to file
results_path = os.path.join(CHECKPOINT_DIR, "head1_test_results.txt")
with open(results_path, "w") as f:
    f.write("HEAD 1 (CONDITION CLASSIFIER) TEST SET RESULTS\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Best epoch: {best_epoch}\n")
    f.write(f"Val macro F1 at best epoch: {checkpoint['val_macro_f1']:.4f}\n\n")
    f.write(f"Test Macro F1:  {test_metrics['macro_f1']:.4f}\n")
    f.write(f"Test Accuracy:  {test_metrics['accuracy']:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(test_metrics["report_str"])
    f.write(f"\nConfusion Matrix:\n")
    f.write(f"Label order: {LABEL_NAMES}\n")
    for row in test_metrics["confusion_matrix"]:
        f.write("  " + "  ".join(f"{v:>5d}" for v in row) + "\n")
    f.write("\n")

print(f"\nResults saved to {results_path}")

## Section 10: Confound Check

This check tests whether the model's mistakes cluster suspiciously by source dataset. If most errors come from one dataset and not the other, it suggests the model is partly learning dataset style instead of actual mental health conditions, which would undermine the results.

We compute accuracy separately for Mukherjee and Dreaddit rows, check the error distribution across sources, and flag if the accuracy gap exceeds 15 percentage points.

In [ ]:
print("=" * 60)
print("SECTION 10: CONFOUND CHECK")
print("=" * 60)

# Build a results dataframe for the test set
confound_df = test_df[["condition_label", "source_dataset"]].copy()
confound_df["true_label"] = test_true
confound_df["predicted_label"] = test_preds
confound_df["predicted_name"] = [ID_TO_LABEL[p] for p in test_preds]
confound_df["correct"] = confound_df["true_label"] == confound_df["predicted_label"]

# Accuracy by source dataset
print("\nAccuracy by source dataset:")
source_accuracies = {}
for source in ["mukherjee", "dreaddit"]:
    mask = confound_df["source_dataset"] == source
    source_sub = confound_df[mask]
    if len(source_sub) == 0:
        print(f"  {source}: no rows in test set")
        continue
    acc = source_sub["correct"].mean()
    n_total = len(source_sub)
    n_correct = int(source_sub["correct"].sum())
    n_errors = n_total - n_correct
    source_accuracies[source] = acc
    print(
        f"  {source:15s}: {acc:.4f} "
        f"({n_correct:,d}/{n_total:,d} correct, {n_errors:,d} errors)"
    )

# Accuracy gap
muk_acc = source_accuracies.get("mukherjee")
dread_acc = source_accuracies.get("dreaddit")

if muk_acc is not None and dread_acc is not None:
    gap = abs(muk_acc - dread_acc) * 100
    print(f"\n  Accuracy gap: {gap:.1f} percentage points")
else:
    gap = None
    print("\n  Cannot compute accuracy gap: missing source in test set")

# Error distribution by source
print("\nError distribution by source:")
errors = confound_df[~confound_df["correct"]]
total_errors = len(errors)
if total_errors > 0:
    for source in ["mukherjee", "dreaddit"]:
        source_errors = (errors["source_dataset"] == source).sum()
        pct = 100 * source_errors / total_errors
        print(f"  {source:15s}: {source_errors:,d} errors ({pct:.1f}% of all errors)")
else:
    print("  No errors (perfect accuracy)")

# Cross-tabulation
print("\nCross-tabulation: source_dataset vs correct/incorrect")
ct = pd.crosstab(
    confound_df["source_dataset"],
    confound_df["correct"].map({True: "Correct", False: "Incorrect"}),
)
print(ct.to_string())

# Final verdict
if gap is not None and gap > 15:
    print(f"\n  WARNING: Confound check FLAGGED a potential issue.")
    print(f"  The accuracy gap between sources ({gap:.1f} pp) exceeds 15 pp.")
    print(f"  The model may be partly learning dataset style instead of condition.")
    print(f"  Investigate before trusting these results.")
    verdict = "FLAGGED"
elif gap is not None:
    print(f"\n  Confound check PASSED.")
    print(f"  The accuracy gap ({gap:.1f} pp) is within acceptable range (15 pp or less).")
    verdict = "PASSED"
else:
    print(f"\n  Confound check INCONCLUSIVE: missing source in test set.")
    verdict = "INCONCLUSIVE"

# Save confound check results
confound_path = os.path.join(CHECKPOINT_DIR, "head1_confound_check.txt")
with open(confound_path, "w") as f:
    f.write("HEAD 1 (CONDITION CLASSIFIER) CONFOUND CHECK\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Verdict: {verdict}\n")
    if gap is not None:
        f.write(f"Accuracy gap: {gap:.1f} percentage points\n")
    f.write(f"\nAccuracy by source:\n")
    if muk_acc is not None:
        f.write(f"  Mukherjee: {muk_acc:.4f}\n")
    if dread_acc is not None:
        f.write(f"  Dreaddit:  {dread_acc:.4f}\n")
    f.write(f"\nError distribution:\n")
    if total_errors > 0:
        for source in ["mukherjee", "dreaddit"]:
            source_errors = (errors["source_dataset"] == source).sum()
            pct = 100 * source_errors / total_errors
            f.write(f"  {source}: {source_errors:,d} errors ({pct:.1f}%)\n")
    else:
        f.write("  No errors\n")
    f.write(f"\nCross-tabulation:\n")
    f.write(ct.to_string())
    f.write("\n")

print(f"\nConfound check results saved to {confound_path}")

## Summary

All steps complete. Output files in `/kaggle/working/checkpoints/`:

| File | Contents |
|------|----------|
| `best_head1_checkpoint.pt` | Best model weights, optimizer state, epoch, label mapping |
| `label_mapping.json` | Integer to class name mapping |
| `head1_test_results.txt` | Full test evaluation results with confusion matrix |
| `head1_confound_check.txt` | Confound check results and verdict |

Download these files from the Kaggle **Output** tab for use in the next steps of the pipeline.

# Section 10.5: Training Curves 

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

epochs = history["epoch"]

fig = plt.figure(figsize=(16, 10))
fig.suptitle("Head 1 — Condition Classifier Training Results", fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

# ── Plot 1: Training Loss ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, history["train_loss"], "o-", color="#e05c5c", linewidth=2, markersize=5)
ax1.set_title("Training Loss per Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_xticks(epochs)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Val Macro F1 + Accuracy ──
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, history["val_macro_f1"], "o-", color="#4c8bf5", linewidth=2, markersize=5, label="Macro F1")
ax2.plot(epochs, history["val_accuracy"], "s--", color="#34a853", linewidth=2, markersize=5, label="Accuracy")
best_ep = history["epoch"][history["val_macro_f1"].index(max(history["val_macro_f1"]))]
ax2.axvline(x=best_ep, color="gray", linestyle=":", linewidth=1.5, label=f"Best epoch ({best_ep})")
ax2.set_title("Val Macro F1 & Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1)
ax2.set_xticks(epochs)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Plot 3: Per-Class F1 over Epochs ──
ax3 = fig.add_subplot(gs[1, 0])
class_colors = {
    "normal": "#34a853", "depression": "#4c8bf5",
    "anxiety": "#fbbc04", "stress": "#e05c5c", "suicidal": "#9b59b6"
}
for cls, color in class_colors.items():
    ax3.plot(epochs, history[f"val_{cls}_f1"], "o-", color=color,
             linewidth=2, markersize=5, label=cls.capitalize())
ax3.axhline(y=0.45, color="red", linestyle=":", linewidth=1, alpha=0.6, label="Stress warn (0.45)")
ax3.set_title("Per-Class F1 over Epochs")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("F1 Score")
ax3.set_ylim(0, 1)
ax3.set_xticks(epochs)
ax3.legend(fontsize=7, ncol=2)
ax3.grid(True, alpha=0.3)

# ── Plot 4: Final Test Confusion Matrix ──
ax4 = fig.add_subplot(gs[1, 1])
cm = test_metrics["confusion_matrix"]
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax4.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)
ax4.set_xticks(range(len(LABEL_NAMES)))
ax4.set_yticks(range(len(LABEL_NAMES)))
ax4.set_xticklabels(LABEL_NAMES, rotation=35, ha="right", fontsize=8)
ax4.set_yticklabels(LABEL_NAMES, fontsize=8)
ax4.set_title("Test Confusion Matrix (Normalized)")
ax4.set_xlabel("Predicted")
ax4.set_ylabel("True")
for i in range(len(LABEL_NAMES)):
    for j in range(len(LABEL_NAMES)):
        val = cm_norm[i, j]
        ax4.text(j, i, f"{val:.2f}", ha="center", va="center",
                 fontsize=7, color="white" if val > 0.6 else "black")

plt.savefig(os.path.join(CHECKPOINT_DIR, "head1_training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to checkpoints/head1_training_curves.png")

In [ ]:
print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

for fname in sorted(os.listdir(CHECKPOINT_DIR)):
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname}  ({size_mb:.2f} MB)")

print(f"\nAll done. Download from the Output tab on Kaggle.")